## Concept 6 — VectorStore Memory

### What is it?
Store past turns as **embeddings** in FAISS. For each new query, retrieve only the
**most semantically relevant** past turns — not the most recent ones.

### Pipeline
```
Storing each turn:
  "Human: ... AI: ..." → embed → store in FAISS

Retrieving for a query:
  query → embed → similarity_search(k=2) → top 2 relevant past turns → LLM → Answer
```

### Why This Beats Chronological Memory
```
Window Memory at Turn 100 : sees only Turns 98-100
VectorStore  at Turn 100  : retrieves Turn 3 (about FAISS) even though it was 97 turns ago
```

### Limitation (what Concept 7 fixes)
Everything so far is in-memory — lost when the process restarts.
→ Concept 7 uses persistent backends (File, SQL) to survive restarts.

In [ ]:
print('hi')

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from MemoryUtils import get_llm, TEST_TURNS, TEST_QUERIES, run_recall_test
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

llm        = get_llm()
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
print('LLM and embeddings ready')

### Step 1 — Build Vector Store from Conversation Turns
Each turn is stored as a single text chunk: `'Human: ... \nAI: ...'`

In [ ]:
# Build text chunks from all TEST_TURNS
turn_texts = [
    f'Human: {human}\nAI: {ai}'
    for human, ai in TEST_TURNS
]

# Create FAISS vector store
vectorstore = FAISS.from_texts(turn_texts, embeddings)

print(f'Stored {len(turn_texts)} turns in FAISS.')
print()
for i, text in enumerate(turn_texts):
    print(f'Turn {i+1}: {text[:80]}...' if len(text) > 80 else f'Turn {i+1}: {text}')

### Step 2 — Semantic Retrieval Per Query
For each query, retrieve the 2 most semantically relevant past turns.

In [ ]:
print('Semantic retrieval — which past turns does each query fetch?')
print('=' * 60)
for query in TEST_QUERIES:
    docs = vectorstore.similarity_search(query, k=2)
    print(f'Query: "{query}"')
    for j, doc in enumerate(docs):
        print(f'  Top {j+1}: {doc.page_content[:90]}...' if len(doc.page_content) > 90 else f'  Top {j+1}: {doc.page_content}')
    print()

### Step 3 — Key Insight: Old but Relevant Turns Are Recalled
Turn 1 is the oldest, but it is retrieved for the name query — window memory would have dropped it.

In [ ]:
docs = vectorstore.similarity_search('What is my name?', k=1)
print('Query: "What is my name?" (info is in Turn 1 — the oldest turn)')
print('Retrieved:')
print(f'  {docs[0].page_content}')
print()
print('VectorStore retrieves Turn 1 even in a 100-turn conversation.')
print('Window memory with k=3 would have dropped Turn 1 after Turn 4.')

### Full Function — chat_with_vectorstore

In [ ]:
# Fresh vector store for the chat function
chat_vs = FAISS.from_texts(
    [f'Human: {h}\nAI: {a}' for h, a in TEST_TURNS],
    embeddings
)

def chat_with_vectorstore(query):
    # Retrieve top-2 semantically relevant past turns
    docs    = chat_vs.similarity_search(query, k=2)
    context = '\n---\n'.join(d.page_content for d in docs)

    messages = [
        SystemMessage(content=f'You are a helpful assistant. Relevant past conversation:\n{context}'),
        HumanMessage(content=query),
    ]
    response = llm.invoke(messages)

    # Store new turn in vector store too
    chat_vs.add_texts([f'Human: {query}\nAI: {response.content}'])
    return response.content

### Test — Recall via Semantic Retrieval

In [ ]:
run_recall_test(chat_with_vectorstore, label='Concept 6 -- VectorStore Memory (FAISS)')